<a href="https://colab.research.google.com/github/eyadarshad/FlyrankAssignment/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eyadarshad/FlyrankAssignment/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

# LANE: Refresh / Content Opportunity Scoring (core lane)
#
# Why this one: the starter pipeline already proved a learned model can beat
# a hand-written rule at surfacing declining pages — the random forest roughly
# tripled precision at the top 50 compared to the "stale × visible" baseline.
# But that baseline is blunt. There's room to build a smarter scoring system
# that combines position drift, CTR, engagement, content age, and word count
# to rank which pages a content team should review first. The output — a
# prioritized refresh queue — is directly actionable, and the cost of getting
# it wrong (wasting editorial hours on healthy pages, or missing pages that
# needed help) is measurable. That makes it a good ML problem.


In [1]:
print("Lane: Refresh / Content Opportunity Scoring")

Lane: Refresh / Content Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

# RESEARCH QUESTION:
# Given observable search and engagement signals, can we rank which pages
# are most likely declining and would benefit from a content refresh —
# better than a simple staleness rule?
#
# UNIT OF ANALYSIS: one content page (one row in the starter CSV).
#
# DECISION: which pages should a content strategist review this sprint,
# out of thousands of candidates?
#
# WHO ACTS: a content/SEO team at a FlyRank client. They can refresh
# maybe 20-50 pages per sprint — the ranking decides which 50 they see first.
#
# OUTPUT: a scored, ranked list with reason codes (e.g. "high impressions
# but falling CTR", "stale content with dropping position").
#
# COST OF A WRONG CALL:
#   False positive (flagging a healthy page) = wasted editorial time,
#   ~2-4 hours per unnecessary refresh. Burns trust in the tool.
#   False negative (missing a declining page) = lost traffic compounds
#   over weeks; harder to recover the longer you wait.
#   A good model needs to be precise at the top of the list.
#
# WHY ML HELPS: a hand rule combines 1-2 signals. A model can weigh
# position, CTR, engagement, age, and word count together — and the
# starter pipeline already showed ~3× better precision. The question is
# whether that holds across clients, not just in-sample.



In [2]:
print("Framing complete — see comments above.")

Framing complete — see comments above.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
y = df["is_declining_label"].values

print(f"Dataset: {df.shape[0]:,} pages x {df.shape[1]} columns, {df['client_id'].nunique()} clients\n")

# NUMBER 1 — how big is the decline problem?
decline_rate = df["is_declining_label"].mean()
n_declining = df["is_declining_label"].sum()
print(f"NUMBER 1 — Decline prevalence:")
print(f"  {n_declining:,} of {len(df):,} pages ({decline_rate:.1%}) are declining.")
print(f"  That means any scoring method needs to do better than a coin flip.\n")

# NUMBER 2 — baseline rule vs learned model (computed live)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Hand rule: stale x visible
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
hand_score = stale * visible * df["impressions_90d"]

# Quick random forest
from sklearn.tree import DecisionTreeClassifier
features = ["content_age_days","days_since_last_update","impressions_90d",
            "avg_position","ctr","word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X, y)
tree_score = tree.predict_proba(X)[:, 1]

base_p50 = precision_at_k(hand_score, y, 50)
tree_p50 = precision_at_k(tree_score, y, 50)
print(f"NUMBER 2 — Precision@50 (of top 50 flagged, how many actually declining?):")
print(f"  Hand rule (stale x visible):  {base_p50:.3f}  -> ~{round(base_p50*50)} correct")
print(f"  Decision tree (depth 3):      {tree_p50:.3f}  -> ~{round(tree_p50*50)} correct")
print(f"  Model improves on the rule — real signal the rule misses.\n")

# NUMBER 3 — staleness alone is weak
stale_pages = df[df["days_since_last_update"] >= 180]
fresh_pages = df[df["days_since_last_update"] < 180]
print(f"NUMBER 3 — Staleness alone as a signal:")
print(f"  Fresh (<180d): {fresh_pages['is_declining_label'].mean():.1%} declining (n={len(fresh_pages):,})")
print(f"  Stale (>=180d): {stale_pages['is_declining_label'].mean():.1%} declining (n={len(stale_pages):,})")
print(f"  Gap is small — staleness alone isn't enough. Need multiple signals combined.")

Dataset: 30,000 pages x 45 columns, 32 clients

NUMBER 1 — Decline prevalence:
  16,262 of 30,000 pages (54.2%) are declining.
  That means any scoring method needs to do better than a coin flip.

NUMBER 2 — Precision@50 (of top 50 flagged, how many actually declining?):
  Hand rule (stale x visible):  0.680  -> ~34 correct
  Decision tree (depth 3):      0.720  -> ~36 correct
  Model improves on the rule — real signal the rule misses.

NUMBER 3 — Staleness alone as a signal:
  Fresh (<180d): 54.2% declining (n=29,826)
  Stale (>=180d): 47.1% declining (n=174)
  Gap is small — staleness alone isn't enough. Need multiple signals combined.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*



In [5]:
# WHAT THIS WORK CAN SAY:
# - "Pages with these signal patterns are more likely to be declining,
#    based on measured trends in the data." (observed, directional)
# - "This model improves precision at the top of the review queue vs
#    the hand-written baseline, measured on held-out client splits."
#    (measured, decision-support)
# - "The ranked list tells a content team where to look first,
#    not what to do." (decision-support, not prescription)
#
# WHAT THIS WORK WILL NEVER SAY:
# - It cannot prove refreshing a page CAUSES recovery. Correlation, not causation.
# - It cannot predict what Google's algorithm will do.
# - It cannot guarantee acting on recommendations improves traffic.
# - All claims use: "observed," "directional," "associated with,"
#   "measured in this dataset" — never "proven" or "will."
#
# DATA LIMITATIONS I'M AWARE OF:
# - 30,000-row snapshot of 32 clients — may not generalize to all clients/time periods.
# - Label (trend_direction == "down") is coarse — a 1% decline and 50% decline
#   get the same label.
# - Missingness is systematic by content_type, not random.

# Quick proof: missingness is systematic
missing = df.groupby("content_type")[["search_volume","word_count"]].apply(
    lambda g: g.isnull().mean()).round(3)
print("Missingness by content_type (confirms it's systematic, not random):\n")
print(missing.to_string())

Missingness by content_type (confirms it's systematic, not random):

                    search_volume  word_count
content_type                                 
comparison article          0.000       0.000
feedly article              1.000       0.000
keyword article             0.014       0.283


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
